In [ ]:
# dependencies
!pip install -q transformers accelerate torch

In [ ]:
# load model
import torch
import re
from transformers import pipeline

model_name = "Qwen/Qwen2-1.5B"

device = 0 if torch.cuda.is_available() else -1

pipe = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=device
)

In [ ]:
# ask model
def ask_model(prompt, max_tokens=50):
    output = pipe(
        prompt,
        max_new_tokens=max_tokens,
        do_sample=False,     # deterministic
        temperature=0.0,
        return_full_text=False
    )

    return output[0]["generated_text"].strip()

In [ ]:
question = "If I have 5 red apples and 7 green apples, how many apples do I have?"

response = ask_model(question)

print("Model Output:")
print(response)

In [ ]:
def extract_number(text):
    numbers = re.findall(r"\d+", text)
    if numbers:
        return int(numbers[-1])
    return None

In [ ]:
def evaluate_math_question(question, correct_answer):
    response = ask_model(question)
    predicted = extract_number(response)

    print("Question:", question)
    print("Model Response:", response)
    print("Extracted Answer:", predicted)
    print("Correct Answer:", correct_answer)

    if predicted == correct_answer:
        print("✅ Correct\n")
        return True
    else:
        print("❌ Incorrect\n")
        return False

In [ ]:
evaluate_math_question(
    "If I have 5 red apples and 7 green apples, how many apples do I have?",
    12
)

In [ ]:
questions = [
    ("If I have 5 red apples and 7 green apples, how many apples do I have?", 12),
    ("John has 10 books and buys 3 more. How many books does he have?", 13),
    ("There are 20 birds, 5 fly away. How many remain?", 15),
    ("Sarah has 8 candies and eats 2. How many are left?", 6),
]

correct = 0

for q, ans in questions:
    if evaluate_math_question(q, ans):
        correct += 1

accuracy = (correct / len(questions))*100
print(f"Final Accuracy: {accuracy:.2f}")

In [ ]:
%pip install datasets

In [ ]:
# Preview Datasets
from datasets import load_dataset

DATASETS = {
    "gsm8k": ("gsm8k", "main"),
    "asdiv": ("yimingzhang/asdiv", None),
    "metamathqa": ("meta-math/MetaMathQA", None),
    "openmathinstruct": ("nvidia/OpenMathInstruct-1", None), # Can change to nvidia/OpenMathInstruct-2 for larger datasets
}

for name, (path, subset) in DATASETS.items():
    print("\n" + "=" * 70)
    print(f"Dataset: {name}")
    print("=" * 70)

    try:
        # Load dataset
        if subset:
            dataset = load_dataset(path, subset)
        else:
            dataset = load_dataset(path)

        print("Available splits:", list(dataset.keys()))

        for split in dataset.keys():
            print(f"\n--- Split: {split} ---")
            print("Number of samples:", len(dataset[split]))
            
            # Column names
            print("Column names:", dataset[split].column_names)
            
            # Feature types (schema)
            print("Features:")
            print(dataset[split].features)

            # Print one example
            print("\nFirst sample:")
            print(dataset[split][0])

    except Exception as e:
        print("Error loading dataset:", e)

In [ ]:
# Configure Datasets
DATASET_CONFIG = {
    "gsm8k": {
        "path": "gsm8k",
        "subset": "main",
        "split": "test",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["answer"]
    },
    "asdiv": {
        "path": "yimingzhang/asdiv",
        "subset": None,
        "split": "test",
        "extract_question": lambda x: x["text"].replace("Question:", "").replace("Answer:", "").strip(),
        "extract_answer": lambda x: x["label"]   # clean numeric answer
    },
    "metamathqa": {
        "path": "meta-math/MetaMathQA",
        "subset": None,
        "split": "train", # No testing dataset available
        "extract_question": lambda x: x["original_question"],
        "extract_answer": lambda x: x["response"]
    },
    "openmathinstruct": {
        "path": "nvidia/OpenMathInstruct-1", # Can change to nvidia/OpenMathInstruct-2 for larger datasets
        "subset": None,
        "split": "validation",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["expected_answer"]
    }
}

In [ ]:
# Build Prompt
def build_prompt(question, name):
    return f"""
You are a deterministic mathematics reasoning engine.

Solve the following math problem step by step using clear logical reasoning.
Show all intermediate reasoning clearly.
At the end, provide the final numerical answer.

Follow these steps:
1. Analyze the problem.
2. Compute step by step.
3. Double-check calculations.
4. Provide final result.

Important rules:
- Do NOT include any text outside the JSON.
- Do NOT use markdown.
- Ensure the final answer is a single number or expression.
- Do NOT restate the instructions.

Output your response strictly in the following JSON format:
{{
  "dataset": "{name}",
  "question": "...",
  "reasoning": "...",
  "final_answer": "..."
}}

Problem:
{question}
"""

In [ ]:
# Generate JSON Output
import json

all_outputs = []

for name, config in DATASET_CONFIG.items():

    print(f"Processing {name}...")

    dataset = load_dataset(
        config["path"],
        config["subset"] if config["subset"] else None
    )

    split_data = dataset[config["split"]]

    for sample in split_data:

        question = config["extract_question"](sample)
        answer = config.get("extract_answer", lambda x: None)(sample)        
        prompt = build_prompt(question, name)

        # ---- send to your model here ----
        # response_text = model.generate(prompt)

        # For demonstration only:
        response = {
            "dataset": name,
            "question": question,
            "reasoning": "model reasoning here",
            "final_answer": "model answer",
            "expencted_answer": answer
        }

        all_outputs.append(response)

        # Optional: limit samples per dataset
        if len(all_outputs) >= 100:
            break

# Save to JSON
with open("math_outputs.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=4, ensure_ascii=False)